In [1]:
import numpy as np
import pylupnt as pnt

In [23]:
pnt.R_MOON, pnt.GM_MOON

(1737.4, 4902.800118)

In [15]:
a = 6541.4  # [km] Semi-major axis
e = 0.6000  # [--] Eccentricity
i = 56.2 * pnt.RAD  # [deg] Inclination
O = 0.00 * pnt.RAD  # [deg] Right ascension of the ascending node
w = 90.0 * pnt.RAD  # [deg] Argument of perigee
M = 0.00 * pnt.RAD  # [deg] Mean anomaly

t_utc = pnt.gregorian2time(2020, 1, 1, 12, 0, 0)
t_tai = pnt.convert_time(t_utc, pnt.UTC, pnt.TAI)

coe0_op = np.array([a, e, i, O, w, M])
rv0_op = pnt.classical2cart(coe0_op, pnt.GM_MOON)
rv0_mci = pnt.convert_frame(t_tai, rv0_op, pnt.MOON_OP, pnt.MOON_CI)
rv0_gcrf = pnt.convert_frame(t_tai, rv0_op, pnt.MOON_OP, pnt.GCRF)
coe_mci = pnt.cart2classical(rv0_mci, pnt.GM_MOON)

coe_mci_tmp = coe_mci.copy()
coe_mci_tmp[2:] = coe_mci_tmp[2:] * pnt.DEG
print(repr(coe_mci_tmp))
print(repr(rv0_mci))

array([ 6.54140000e+03,  6.00000000e-01,  6.08802626e+01,  8.48999999e+01,
        1.16785933e+02, -1.15020532e-15])
array([-1.23700409e+03, -1.07346186e+03,  2.04056040e+03,  2.40818852e-01,
       -1.57331956e+00, -6.81677756e-01])


In [24]:
Dt = 20 * pnt.SECS_MINUTE
t_final = 200 * pnt.SECS_DAY
N_steps = int(t_final / Dt)

rv0 = rv0_mci
t0 = t_tai
tfs = t0 + np.arange(0, N_steps + 1) * Dt

params = pnt.IntegratorParams()
params.max_iter = 10
params.abstol = 1e-8
params.reltol = 1e-8

dyn = pnt.NBodyDynamics(pnt.PD45)
dyn.add_body(pnt.Body.Moon(20, 20))
# dyn.add_body(pnt.Body.Earth())
# dyn.add_body(pnt.Body.Sun())
dyn.set_frame(pnt.MOON_CI)
dyn.set_time_step(60.0)
dyn.set_integrator_params(params)

In [25]:
import time
t_start = time.time()
rvs = dyn.propagate(rv0, t0, tfs)
t_elapsed = time.time() - t_start
print(f"Elapsed time: {t_elapsed:.3f} s")
print(f"N_steps: {N_steps}")

Elapsed time: 2.360 s
N_steps: 14400


In [26]:
from tqdm import tqdm

rvs = np.zeros((N_steps + 1, 6))
rvs[0] = rv0
t_start = time.time()
for i in tqdm(range(1, N_steps + 1)):
    rvs[i] = dyn.propagate(rvs[i - 1], tfs[i - 1], tfs[i])
t_elapsed = time.time() - t_start
print(f"Elapsed time: {t_elapsed:.3f} s")
print(f"N_steps: {N_steps}")

100%|██████████| 14400/14400 [00:02<00:00, 5973.05it/s]

Elapsed time: 2.415 s
N_steps: 14400


In [27]:
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = "plotly_white"

n = 1
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=rvs[::n, 0], y=rvs[::n, 1], z=rvs[::n, 2], mode='lines', name='Orbit'))
fig.update_layout(scene=dict(aspectmode='data'))
fig.show()
